In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table_name = (
    "global_economic_analytics.bronze.world_bank_raw"
)

silver_table_name = (
    "global_economic_analytics.silver.macro_observation"
)

gold_table_name = (
    "global_economic_analytics.gold.country_year_macro"
)

print(bronze_table_name)
print(silver_table_name)
print(gold_table_name)

In [0]:
df_bronze = spark.table(bronze_table_name)
df_silver = spark.table(silver_table_name)
df_gold = spark.table(gold_table_name)

In [0]:
bronze_count = df_bronze.count()
silver_count = df_silver.count()
gold_count = df_gold.count()

print(f"Bronze: {bronze_count}")
print(f"Silver: {silver_count}")
print(f"Gold: {gold_count}")

In [0]:
assert bronze_count == 1, (
    f"Esperado 1 registro Bronze, encontrado: {bronze_count}"
)

assert silver_count == 25, (
    f"Esperados 25 registros Silver, encontrados: {silver_count}"
)

assert gold_count == 25, (
    f"Esperados 25 registros Gold, encontrados: {gold_count}"
)

print("Contagens das camadas validadas com sucesso!")

In [0]:
bronze_duplicate_count = (
    df_bronze
    .groupBy("idempotency_key")
    .count()
    .filter(
        F.col("idempotency_key").isNotNull()
        & (F.col("count") > 1)
    )
    .count()
)

print(bronze_duplicate_count)

In [0]:
assert bronze_duplicate_count == 0, (
    f"Foram encontradas {bronze_duplicate_count} "
    "chaves de idempotência duplicadas na Bronze"
)

print("Unicidade da camada Bronze validada com sucesso!")

In [0]:
bronze_null_required_count = (
    df_bronze
    .filter(
        F.col("ingestion_id").isNull()
        | F.col("ingestion_timestamp").isNull()
        | F.col("country_code").isNull()
        | F.col("indicator_code").isNull()
        | F.col("http_status_code").isNull()
        | F.col("raw_payload").isNull()
        | F.col("idempotency_key").isNull()
    )
    .count()
)

print(bronze_null_required_count)

In [0]:
assert bronze_null_required_count == 0, (
    f"Foram encontrados {bronze_null_required_count} "
    "registros com campos obrigatórios nulos na Bronze"
)

print("Campos obrigatórios da camada Bronze validados com sucesso!")

In [0]:
silver_duplicate_count = (
    df_silver
    .groupBy("observation_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(silver_duplicate_count)

In [0]:
assert silver_duplicate_count == 0, (
    f"Foram encontradas {silver_duplicate_count} "
    "chaves de observação duplicadas na Silver"
)

print("Unicidade da camada Silver validada com sucesso!")

In [0]:
silver_null_required_count = (
    df_silver
    .filter(
        F.col("observation_key").isNull()
        | F.col("country_code").isNull()
        | F.col("country_name").isNull()
        | F.col("indicator_code").isNull()
        | F.col("indicator_name").isNull()
        | F.col("year").isNull()
        | F.col("indicator_value").isNull()
        | F.col("ingestion_id").isNull()
        | F.col("ingestion_timestamp").isNull()
    )
    .count()
)

print(silver_null_required_count)

In [0]:
assert silver_null_required_count == 0, (
    f"Foram encontrados {silver_null_required_count} "
    "registros com campos obrigatórios nulos na Silver"
)

print("Campos obrigatórios da camada Silver validados com sucesso!")

In [0]:
silver_invalid_year_count = (
    df_silver
    .filter(
        (F.col("year") < 2000)
        | (F.col("year") > 2024)
    )
    .count()
)

print(silver_invalid_year_count)

In [0]:
assert silver_invalid_year_count == 0, (
    f"Foram encontrados {silver_invalid_year_count} "
    "registros Silver fora do intervalo de 2000 a 2024"
)

print("Intervalo de anos da camada Silver validado com sucesso!")

In [0]:
silver_invalid_country_code_count = (
    df_silver
    .filter(
        F.length(F.trim(F.col("country_code"))) != 3
    )
    .count()
)

print(silver_invalid_country_code_count)

In [0]:
assert silver_invalid_country_code_count == 0, (
    f"Foram encontrados {silver_invalid_country_code_count} "
    "registros Silver com código de país inválido"
)

print("Códigos de país da camada Silver validados com sucesso!")

In [0]:
silver_invalid_country_code_format_count = (
    df_silver
    .filter(
        ~F.col("country_code").rlike("^[A-Z]{3}$")
    )
    .count()
)

print(silver_invalid_country_code_format_count)

In [0]:
assert silver_invalid_country_code_format_count == 0, (
    f"Foram encontrados {silver_invalid_country_code_format_count} "
    "registros Silver fora do padrão ISO 3"
)

print("Formato dos códigos de país validado com sucesso!")

In [0]:
gold_duplicate_count = (
    df_gold
    .groupBy("country_year_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(gold_duplicate_count)

In [0]:
assert gold_duplicate_count == 0, (
    f"Foram encontradas {gold_duplicate_count} "
    "chaves de país e ano duplicadas na Gold"
)

print("Unicidade da camada Gold validada com sucesso!")

In [0]:
gold_null_required_count = (
    df_gold
    .filter(
        F.col("country_year_key").isNull()
        | F.col("country_code").isNull()
        | F.col("country_name").isNull()
        | F.col("year").isNull()
        | F.col("gdp_current_usd").isNull()
        | F.col("gdp_current_billion_usd").isNull()
    )
    .count()
)

print(gold_null_required_count)

In [0]:
assert gold_null_required_count == 0, (
    f"Foram encontrados {gold_null_required_count} "
    "registros com campos obrigatórios nulos na Gold"
)

print("Campos obrigatórios da camada Gold validados com sucesso!")

In [0]:
gold_invalid_gdp_count = (
    df_gold
    .filter(F.col("gdp_current_usd") <= 0)
    .count()
)

print(gold_invalid_gdp_count)

In [0]:
assert gold_invalid_gdp_count == 0, (
    f"Foram encontrados {gold_invalid_gdp_count} "
    "registros Gold com PIB menor ou igual a zero"
)

print("Valores de PIB da camada Gold validados com sucesso!")

In [0]:
gold_inconsistent_gdp_conversion_count = (
    df_gold
    .filter(
        F.col("gdp_current_billion_usd")
        != F.round(
            F.col("gdp_current_usd") / F.lit(1_000_000_000),
            2
        )
    )
    .count()
)

print(gold_inconsistent_gdp_conversion_count)

In [0]:
assert gold_inconsistent_gdp_conversion_count == 0, (
    f"Foram encontrados {gold_inconsistent_gdp_conversion_count} "
    "registros Gold com conversão inconsistente do PIB"
)

print("Conversão do PIB em bilhões validada com sucesso!")

In [0]:
gold_year_summary = (
    df_gold
    .groupBy("country_code")
    .agg(
        F.min("year").alias("minimum_year"),
        F.max("year").alias("maximum_year"),
        F.countDistinct("year").alias("distinct_year_count")
    )
    .withColumn(
        "expected_year_count",
        F.col("maximum_year") - F.col("minimum_year") + 1
    )
    .withColumn(
        "missing_year_count",
        F.col("expected_year_count") - F.col("distinct_year_count")
    )
)

display(gold_year_summary)

In [0]:
gold_country_with_missing_years_count = (
    gold_year_summary
    .filter(F.col("missing_year_count") != 0)
    .count()
)

assert gold_country_with_missing_years_count == 0, (
    f"Foram encontrados {gold_country_with_missing_years_count} "
    "países com anos ausentes na camada Gold"
)

print("Continuidade anual da camada Gold validada com sucesso!")

In [0]:
silver_gold_count_difference = silver_count - gold_count

print(silver_gold_count_difference)

In [0]:
assert silver_gold_count_difference == 0, (
    f"Diferença inesperada entre Silver e Gold: "
    f"{silver_gold_count_difference}"
)

print("Consistência entre Silver e Gold validada com sucesso!")

In [0]:
silver_country_year_count = (
    df_silver
    .select(
        "country_code",
        "year"
    )
    .distinct()
    .count()
)

gold_country_year_count = (
    df_gold
    .select(
        "country_code",
        "year"
    )
    .distinct()
    .count()
)

print(f"Silver país-ano: {silver_country_year_count}")
print(f"Gold país-ano: {gold_country_year_count}")

In [0]:
assert silver_country_year_count == gold_country_year_count, (
    "A quantidade de combinações país-ano diverge "
    f"entre Silver ({silver_country_year_count}) "
    f"e Gold ({gold_country_year_count})"
)

print("Granularidade país-ano validada entre Silver e Gold!")

In [0]:
silver_country_year = (
    df_silver
    .select(
        "country_code",
        "year"
    )
    .distinct()
)

gold_country_year = (
    df_gold
    .select(
        "country_code",
        "year"
    )
    .distinct()
)

missing_country_year_in_gold = (
    silver_country_year
    .join(
        gold_country_year,
        on=["country_code", "year"],
        how="left_anti"
    )
)

missing_country_year_count = (
    missing_country_year_in_gold.count()
)

print(missing_country_year_count)

In [0]:
assert missing_country_year_count == 0, (
    f"Foram encontradas {missing_country_year_count} "
    "combinações país-ano da Silver ausentes na Gold"
)

print("Cobertura da Silver na Gold validada com sucesso!")

In [0]:
extra_country_year_in_gold = (
    gold_country_year
    .join(
        silver_country_year,
        on=["country_code", "year"],
        how="left_anti"
    )
)

extra_country_year_count = (
    extra_country_year_in_gold.count()
)

print(extra_country_year_count)

In [0]:
assert extra_country_year_count == 0, (
    f"Foram encontradas {extra_country_year_count} "
    "combinações país-ano na Gold sem correspondência na Silver"
)

print("Ausência de registros extras na Gold validada com sucesso!")

In [0]:
df_silver_gdp = (
    df_silver
    .filter(
        F.col("indicator_code") == "NY.GDP.MKTP.CD"
    )
    .select(
        "country_code",
        "year",
        F.col("indicator_value").alias("silver_gdp_usd")
    )
)

df_gold_gdp = (
    df_gold
    .select(
        "country_code",
        "year",
        F.col("gdp_current_usd").alias("gold_gdp_usd")
    )
)

gdp_value_mismatch_count = (
    df_silver_gdp
    .join(
        df_gold_gdp,
        on=["country_code", "year"],
        how="inner"
    )
    .filter(
        ~F.col("silver_gdp_usd").eqNullSafe(
            F.col("gold_gdp_usd")
        )
    )
    .count()
)

print(gdp_value_mismatch_count)

In [0]:
assert gdp_value_mismatch_count == 0, (
    f"Foram encontradas {gdp_value_mismatch_count} "
    "divergências de PIB entre Silver e Gold"
)

print("Valores de PIB entre Silver e Gold validados com sucesso!")

In [0]:
quality_results = [
    ("Bronze", "Contagem esperada", bronze_count == 1),
    ("Bronze", "Unicidade da idempotency_key", bronze_duplicate_count == 0),
    ("Bronze", "Campos obrigatórios preenchidos", bronze_null_required_count == 0),

    ("Silver", "Contagem esperada", silver_count == 25),
    ("Silver", "Unicidade da observation_key", silver_duplicate_count == 0),
    ("Silver", "Campos obrigatórios preenchidos", silver_null_required_count == 0),
    ("Silver", "Anos dentro do intervalo", silver_invalid_year_count == 0),
    ("Silver", "Formato ISO 3 do país", silver_invalid_country_code_format_count == 0),

    ("Gold", "Contagem esperada", gold_count == 25),
    ("Gold", "Unicidade da country_year_key", gold_duplicate_count == 0),
    ("Gold", "Campos obrigatórios preenchidos", gold_null_required_count == 0),
    ("Gold", "PIB maior que zero", gold_invalid_gdp_count == 0),
    ("Gold", "Conversão do PIB em bilhões", gold_inconsistent_gdp_conversion_count == 0),
    ("Gold", "Continuidade anual", gold_country_with_missing_years_count == 0),

    ("Silver-Gold", "Cobertura país-ano", missing_country_year_count == 0),
    ("Silver-Gold", "Ausência de registros extras", extra_country_year_count == 0),
    ("Silver-Gold", "Consistência dos valores do PIB", gdp_value_mismatch_count == 0)
]

df_quality_results = spark.createDataFrame(
    quality_results,
    ["data_layer", "quality_rule", "passed"]
)

display(df_quality_results)

In [0]:
quality_summary = (
    df_quality_results
    .agg(
        F.count("*").alias("total_rules"),
        F.sum(
            F.col("passed").cast("int")
        ).alias("passed_rules"),
        F.sum(
            (~F.col("passed")).cast("int")
        ).alias("failed_rules")
    )
    .first()
)

print(quality_summary)

In [0]:
from datetime import datetime, timezone

quality_run_timestamp = datetime.now(timezone.utc)
quality_run_id = quality_run_timestamp.strftime(
    "%Y%m%dT%H%M%S%fZ"
)

print(quality_run_id)
print(quality_run_timestamp)

In [0]:
df_quality_results_enriched = (
    df_quality_results
    .withColumn(
        "quality_run_id",
        F.lit(quality_run_id)
    )
    .withColumn(
        "quality_run_timestamp",
        F.lit(quality_run_timestamp)
    )
    .select(
        "quality_run_id",
        "quality_run_timestamp",
        "data_layer",
        "quality_rule",
        "passed"
    )
)

df_quality_results_enriched.printSchema()

In [0]:
quality_table_name = (
    "global_economic_analytics.gold.data_quality_result"
)

print(quality_table_name)

In [0]:
quality_table_exists = spark.catalog.tableExists(
    quality_table_name
)

print(quality_table_exists)

In [0]:
(
    df_quality_results_enriched.write
    .format("delta")
    .saveAsTable(quality_table_name)
)

print("Tabela de resultados de qualidade criada com sucesso!")

In [0]:
quality_table_exists_after_write = spark.catalog.tableExists(
    quality_table_name
)

print(quality_table_exists_after_write)

In [0]:
quality_table_count = spark.table(
    quality_table_name
).count()

print(quality_table_count)

In [0]:
display(
    spark.table(quality_table_name)
    .groupBy("quality_run_id")
    .agg(
        F.count("*").alias("total_rules"),
        F.sum(
            F.col("passed").cast("int")
        ).alias("passed_rules"),
        F.sum(
            (~F.col("passed")).cast("int")
        ).alias("failed_rules")
    )
)

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {quality_table_name}
    IS 'Histórico dos resultados das regras de qualidade executadas nas camadas do projeto.'
""")

print("Comentário da tabela de qualidade adicionado com sucesso!")

In [0]:
quality_table_comment = spark.sql("""
    SELECT comment
    FROM global_economic_analytics.information_schema.tables
    WHERE table_schema = 'gold'
      AND table_name = 'data_quality_result'
""").first()["comment"]

print(quality_table_comment)

In [0]:
spark.sql(f"""
    ALTER TABLE {quality_table_name}
    ALTER COLUMN quality_run_id
    COMMENT 'Identificador único da execução das regras de qualidade.'
""")

print("Coluna quality_run_id documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {quality_table_name}
    ALTER COLUMN quality_run_timestamp
    COMMENT 'Data e hora em UTC da execução das regras de qualidade.'
""")

print("Coluna quality_run_timestamp documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {quality_table_name}
    ALTER COLUMN data_layer
    COMMENT 'Camada ou relacionamento entre camadas avaliado pela regra de qualidade.'
""")

print("Coluna data_layer documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {quality_table_name}
    ALTER COLUMN quality_rule
    COMMENT 'Nome da regra de qualidade avaliada.'
""")

print("Coluna quality_rule documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {quality_table_name}
    ALTER COLUMN passed
    COMMENT 'Indica se a regra de qualidade foi aprovada.'
""")

print("Coluna passed documentada com sucesso!")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE TABLE {quality_table_name}
    """)
)

In [0]:
spark.sql(f"""
    ALTER TABLE {quality_table_name}
    SET TBLPROPERTIES (
        'project' = 'global_economic_analytics',
        'data_layer' = 'gold',
        'table_purpose' = 'data quality monitoring',
        'data_granularity' = 'one row per quality rule and execution'
    )
""")

print("Propriedades da tabela de qualidade adicionadas com sucesso!")

In [0]:
quality_table_properties = spark.sql(f"""
    SHOW TBLPROPERTIES {quality_table_name}
""")

display(
    quality_table_properties.filter(
        """
        key IN (
            'project',
            'data_layer',
            'table_purpose',
            'data_granularity'
        )
        """
    )
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE DETAIL {quality_table_name}
    """).select(
        "format",
        "name",
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {quality_table_name}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
)

In [0]:
existing_quality_run_count = (
    spark.table(quality_table_name)
    .filter(
        F.col("quality_run_id") == quality_run_id
    )
    .count()
)

print(existing_quality_run_count)

In [0]:
def quality_run_already_exists(table_name, run_id):
    return (
        spark.table(table_name)
        .filter(F.col("quality_run_id") == run_id)
        .limit(1)
        .count()
        > 0
    )

In [0]:
quality_run_exists = quality_run_already_exists(
    quality_table_name,
    quality_run_id
)

print(quality_run_exists)

In [0]:
def write_quality_results(df, table_name, run_id):
    if quality_run_already_exists(table_name, run_id):
        print(
            f"Execução {run_id} já registrada. "
            "Gravação ignorada."
        )
        return False

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

    print(f"Execução {run_id} gravada com sucesso.")
    return True

In [0]:
quality_write_result = write_quality_results(
    df_quality_results_enriched,
    quality_table_name,
    quality_run_id
)

print(quality_write_result)

In [0]:
quality_count_after_idempotency_test = spark.table(
    quality_table_name
).count()

print(quality_count_after_idempotency_test)

In [0]:
quality_final_validation = (
    spark.table(quality_table_name)
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("quality_run_id").alias("total_runs"),
        F.countDistinct("quality_rule").alias("distinct_rules"),
        F.sum(
            F.col("passed").cast("int")
        ).alias("passed_records"),
        F.sum(
            (~F.col("passed")).cast("int")
        ).alias("failed_records")
    )
    .first()
)

print(quality_final_validation)

In [0]:
distinct_layer_rule_count = (
    spark.table(quality_table_name)
    .select(
        "data_layer",
        "quality_rule"
    )
    .distinct()
    .count()
)

print(distinct_layer_rule_count)

In [0]:
assert quality_final_validation["total_records"] == 17
assert quality_final_validation["total_runs"] == 1
assert distinct_layer_rule_count == 17
assert quality_final_validation["passed_records"] == 17
assert quality_final_validation["failed_records"] == 0

print("Tabela de resultados de qualidade validada com sucesso!")